### Implementing simple Chatbot Using LangGraph

In [ ]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

## Reducers
from typing import Annotated
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")
llm.invoke("Hello")

In [ ]:
# from langchain_groq import ChatGroq

# llm_groq=ChatGroq(model="qwen-14b")
# llm_groq.invoke("Hey I am Shailendra and i like to play cricket")

### We Will start With Creating Nodes

In [ ]:
def superbot(state:State):
    return {"messages":[llm.invoke(state['messages'])]}

In [ ]:
graph=StateGraph(State)

## node
graph.add_node("SuperBot",superbot)
## Edges

graph.add_edge(START,"SuperBot")
graph.add_edge("SuperBot",END)


graph_builder=graph.compile()


## Display
from IPython.display import Image, display
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [32]:
## Invocation

graph_builder.invoke({'messages':"Hi,My name is Shailendra And I like cricket"})
graph_builder.invoke({'messages':"Give me best player in criket"})
graph_builder.invoke({'messages':"any amazing world record, do you know?"})

{'messages': [HumanMessage(content='any amazing world record, do you know?', additional_kwargs={}, response_metadata={}, id='46dbbc43-fd2f-4107-b657-2867b6831fc8'),
  AIMessage(content="Yes, there have been many amazing world records across various fields. Here are a few notable ones:\n\n1. **Fastest Marathon**: As of October 2023, Eliud Kipchoge holds the men's marathon world record with a time of 2 hours, 1 minute, and 9 seconds, achieved at the 2022 Berlin Marathon.\n\n2. **Tallest Building**: The Burj Khalifa in Dubai, United Arab Emirates, holds the record for the tallest building in the world, standing at 828 meters (2,717 feet) tall.\n\n3. **Longest Freefall**: On October 14, 2012, Felix Baumgartner set the record for the highest freefall jump at 38,969 meters (127,852 feet) from the stratosphere, as part of the Red Bull Stratos project.\n\n4. **Deepest Ocean Dive**: In 2019, explorer Victor Vescovo set a world record by reaching the bottom of the Mariana Trench—the deepest know

#### Streaming The responses

In [36]:
for event in graph_builder.stream({"messages":"Hello My name is Shailendra"}, stream_mode="values"):
    print(event)

{'messages': [HumanMessage(content='Hello My name is Shailendra', additional_kwargs={}, response_metadata={}, id='ff087385-c4c5-4e5f-9e19-4baae8005f95')]}
{'messages': [HumanMessage(content='Hello My name is Shailendra', additional_kwargs={}, response_metadata={}, id='ff087385-c4c5-4e5f-9e19-4baae8005f95'), AIMessage(content='Hello Shailendra! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 14, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f33640a400', 'id': 'chatcmpl-CLWhEOA0poD3jCXnkdq3hL6gIQPCP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--84720d54-0554-489b-80d6-b481567ccfea-0', usage_metadata={'input_tokens': 14, '